<a href="https://colab.research.google.com/github/Ashprakash/groundfin/blob/main/benchmark/financebench_colab_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>\n\n# FinanceBench Pilot for Grounded Execution Precision\n\nGoal: load FinanceBench, inspect task structure, run first baseline numbers, and prepare data artifacts for a grounded distillation / counterfactual benchmark paper.\n\nPrimary dataset: `PatronusAI/financebench` on Hugging Face. The FinanceBench paper states that the benchmark contains 10,231 financial QA examples and that the open-source cases are available publicly. The Hugging Face dataset currently exposes the 150-row open subset.\n\nResearch question for this pilot:\n\n> Do existing finance QA examples let us separate parametric guessing from evidence-conditioned answering, and can we construct counterfactual/missing-evidence tests that expose that difference?

In [ ]:
!pip -q install datasets pandas numpy scikit-learn tqdm openai

In [ ]:
import os
import re
import json
import math
import textwrap
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 220)

## 1. Load FinanceBench

In [ ]:
DATASET_ID = 'PatronusAI/financebench'
ds = load_dataset(DATASET_ID, split='train')
df = ds.to_pandas()

print(df.shape)
display(df.head(3))
print(df.columns.tolist())

## 2. Inspect Task Distribution\n\nThese are the first numbers we need for the benchmark paper: how many examples, question types, reasoning types, companies, years, and document types.

In [ ]:
for col in ['question_type', 'question_reasoning', 'dataset_subset_label', 'gics_sector', 'doc_type', 'doc_period']:
    if col in df.columns:
        print(f'\\n{col}')
        display(df[col].value_counts(dropna=False).head(20).to_frame('count'))

print('Unique companies:', df['company'].nunique())
display(df['company'].value_counts().head(20).to_frame('count'))

## 3. Normalize Evidence Bundles\n\nFinanceBench includes evidence objects. We flatten them into compact evidence text for prompting and later evidence-support scoring.

In [ ]:
def flatten_evidence(evidence):
    if evidence is None or (isinstance(evidence, float) and math.isnan(evidence)):
        return ''
    if isinstance(evidence, str):
        try:
            evidence = json.loads(evidence)
        except Exception:
            return evidence
    parts = []
    for ev in evidence:
        if not isinstance(ev, dict):
            parts.append(str(ev))
            continue
        doc = ev.get('doc_name', '')
        page = ev.get('evidence_page_num', '')
        txt = ev.get('evidence_text', '') or ev.get('evidence_text_full_page', '')
        prefix = f'[doc={doc} page={page}]'.strip()
        parts.append(f'{prefix} {txt}'.strip())
    return '\\n\\n'.join(parts)

df['evidence_text'] = df['evidence'].apply(flatten_evidence)
df['evidence_chars'] = df['evidence_text'].str.len()
display(df[['financebench_id', 'question', 'answer', 'evidence_chars', 'evidence_text']].head(3))
display(df['evidence_chars'].describe().to_frame())

## 4. Create Prompt Variants\n\nThe key experiment is to compare the same model with and without grounded evidence. Later, GEP-style experiments test typed extraction, execution, verification, and selective commitment.

In [ ]:
SYSTEM_INSTRUCTION = '''You are a careful financial QA assistant. Answer only from the provided evidence when evidence is provided. If the answer is not supported, say INSUFFICIENT_EVIDENCE. Return a concise answer and a confidence from 0 to 1.'''

def make_prompt(row, with_evidence=True):
    if with_evidence:
        return f'''{SYSTEM_INSTRUCTION}\n\nQuestion:\n{row['question']}\n\nEvidence:\n{row['evidence_text']}\n\nReturn JSON with keys: answer, confidence, evidence_support, short_rationale.'''
    return f'''{SYSTEM_INSTRUCTION}\n\nQuestion:\n{row['question']}\n\nNo evidence is provided.\n\nReturn JSON with keys: answer, confidence, evidence_support, short_rationale.'''

sample = df.iloc[0]
print(make_prompt(sample, with_evidence=True)[:2000])

## 5. Simple Scoring Utilities\n\nThis is deliberately lightweight for pilot work. For paper experiments, we should use exact numeric parsing, unit normalization, semantic judge scoring, and human validation on a subset.

In [ ]:
def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r'[$,%]', '', s)
    s = re.sub(r'\\s+', ' ', s)
    return s

def extract_numbers(s):
    return [float(x.replace(',', '')) for x in re.findall(r'-?\\d+(?:,\\d{3})*(?:\\.\\d+)?', str(s))]

def numeric_close(pred, gold, rel_tol=0.02, abs_tol=0.05):
    pnums = extract_numbers(pred)
    gnums = extract_numbers(gold)
    if not pnums or not gnums:
        return False
    for p in pnums:
        for g in gnums:
            if abs(p - g) <= max(abs_tol, rel_tol * max(1.0, abs(g))):
                return True
    return False

def weak_answer_match(pred, gold):
    pn = normalize_text(pred)
    gn = normalize_text(gold)
    if gn and (gn in pn or pn in gn):
        return True
    return numeric_close(pred, gold)

tests = [('$1577.00', '$1,577'), ('24.26', '24.2'), ('No', 'Yes')]
for p, g in tests:
    print(p, g, weak_answer_match(p, g))

## 6. Optional API Baseline\n\nSet `OPENAI_API_KEY` in Colab secrets or environment if you want to run this. Keep `N_EXAMPLES` small for the pilot.\n\nWe run two conditions:\n\n- `question_only`: tests parametric guessing / refusal behavior.\n- `with_gold_evidence`: tests evidence-conditioned answering with the gold evidence bundle.\n\nFor the paper, these become baselines against the distilled student.

In [ ]:
USE_OPENAI = bool(os.environ.get('OPENAI_API_KEY'))
MODEL = 'gpt-4.1-mini'
N_EXAMPLES = 20

def call_openai(prompt, model=MODEL):
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

if not USE_OPENAI:
    print('OPENAI_API_KEY not set. Skipping API baseline.')
else:
    rows = []
    pilot = df.sample(min(N_EXAMPLES, len(df)), random_state=7).reset_index(drop=True)
    for _, row in tqdm(pilot.iterrows(), total=len(pilot)):
        for condition, with_evidence in [('question_only', False), ('with_gold_evidence', True)]:
            pred = call_openai(make_prompt(row, with_evidence=with_evidence))
            rows.append({
                'financebench_id': row['financebench_id'],
                'condition': condition,
                'question': row['question'],
                'gold_answer': row['answer'],
                'prediction': pred,
                'weak_match': weak_answer_match(pred, row['answer']),
            })
    results = pd.DataFrame(rows)
    display(results.head())
    display(results.groupby('condition')['weak_match'].mean().to_frame('weak_accuracy'))

## 7. Candidate Counterfactual Transformations\n\nThis cell does not generate final benchmark examples yet. It identifies examples that are likely good candidates for GEP-Bench counterfactual splits.

In [ ]:
def has_numeric_answer(answer):
    return len(extract_numbers(answer)) > 0

df['has_numeric_answer'] = df['answer'].apply(has_numeric_answer)
df['candidate_numeric_counterfactual'] = df['has_numeric_answer'] & (df['evidence_chars'] > 200)
df['candidate_missing_evidence'] = df['evidence_chars'] > 200
df['candidate_temporal'] = df['question'].str.contains('FY|Q[1-4]|year|2022|2023|2019|2018', case=False, regex=True, na=False)

summary = df[['candidate_numeric_counterfactual', 'candidate_missing_evidence', 'candidate_temporal']].sum().to_frame('count')
display(summary)
display(df.loc[df['candidate_numeric_counterfactual'], ['financebench_id', 'company', 'doc_name', 'question', 'answer']].head(20))

## 8. Save Pilot Artifacts\n\nThis creates CSV/JSONL files that can feed the benchmark and method folders.

In [ ]:
pilot_export = df[[
    'financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning',
    'question', 'answer', 'justification', 'evidence_text', 'gics_sector', 'doc_type', 'doc_period',
    'candidate_numeric_counterfactual', 'candidate_missing_evidence', 'candidate_temporal'
]].copy()

pilot_export.to_csv('financebench_pilot_flat.csv', index=False)
pilot_export.to_json('financebench_pilot_flat.jsonl', orient='records', lines=True)
print('Wrote financebench_pilot_flat.csv and financebench_pilot_flat.jsonl')

## Next Experiments\n\n1. Run question-only vs gold-evidence baseline on 20-50 examples.\n2. Inspect failures manually and label whether errors are retrieval, arithmetic, grounding, or refusal failures.\n3. Create 30-50 counterfactual examples from numeric/table questions.\n4. Test whether a model that answers original examples fails when evidence values change.\n5. Use teacher outputs to create GEP supervision traces: answer distribution, confidence, evidence support, abstention label.\n6. Decide whether first student experiment is prompt-only, LoRA fine-tuning, or both.